In [ ]:

import sys
import os
print(os.getcwd())
# adjust this path to your project root
PROJECT_ROOT = os.path.abspath("../src/")  # or "../.." depending on where notebook lives

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Added to path:", PROJECT_ROOT)

In [ ]:
import json
import pandas as pd
from datetime import datetime, timedelta, timezone

from extract import make_client
from pendo.endpoint_factory import get_endpoint
from pendo.endpoints import AggregationEndpoint

def dt_to_ms(dt: datetime) -> int:
    return int(dt.timestamp() * 1000)

def pretty(obj):
    print(json.dumps(obj, indent=2, default=str))

client = make_client()
endpoint = get_endpoint("aggregations", client)
assert isinstance(endpoint, AggregationEndpoint)

# test window
end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=14)
first_ms = dt_to_ms(end_dt)
days = (end_dt - start_dt).days
TEST_APP_ID = 5821761749057536  # replace if needed

def test_source(source_name, extra_source=None, filter_expr=None, limit=5):
    source_obj = {
        "timeSeries": {
            "period": "dayRange",
            "first": first_ms,
            "count": -days,
        }
    }
    source_obj[source_name] = extra_source

    payload = {
        "response": {"mimeType": "application/json"},
        "request": {
            "pipeline": [
                {"source": source_obj},
                *([{"filter": filter_expr}] if filter_expr else []),
                {"limit": limit},
            ]
        },
    }

    print(f"\n=== TEST SOURCE: {source_name} ===")
    try:
        resp = endpoint.run(payload)
        pretty(resp)
        return resp
    except Exception as e:
        print(type(e).__name__, e)
        return None

In [ ]:
resp_page = test_source("pageEvents", {"appId": TEST_APP_ID})

In [ ]:
payload_active_users = {
    "response": {"mimeType": "application/json"},
    "request": {
        "pipeline": [
            {
                "source": {
                    "timeSeries": {
                        "period": "dayRange",
                        "first": first_ms,
                        "count": -days,
                    },
                    "pageEvents": {"appId": TEST_APP_ID},
                }
            },
            {
                "group": {
                    "group": ["appId"],
                    "fields": {
                        "activeUsers": {"countDistinct": "visitorId"},
                        "eventCount": {"count": None},
                    },
                }
            },
        ]
    },
}

resp_active_users = endpoint.run(payload_active_users)
pretty(resp_active_users)

In [ ]:
payload_unique_visitors = {
    "response": {"mimeType": "application/json"},
    "request": {
        "pipeline": [
            {
                "source": {
                    "timeSeries": {
                        "period": "dayRange",
                        "first": first_ms,
                        "count": -days,
                    },
                    "pageEvents": {"appId": TEST_APP_ID},
                }
            },
            {
                "group": {
                    "group": ["appId", "visitorId"],
                    "fields": {
                        "eventCount": {"count": None},
                        "firstTime": {"min": "firstTime"},
                        "lastTime": {"max": "lastTime"},
                    },
                }
            },
            {"limit": 100000}
        ]
    },
}

resp_unique_visitors = endpoint.run(payload_unique_visitors)
pretty(resp_unique_visitors)

In [ ]:
def results_to_df(resp):
    if isinstance(resp, dict) and "results" in resp and isinstance(resp["results"], list):
        return pd.json_normalize(resp["results"])
    if isinstance(resp, list):
        return pd.json_normalize(resp)
    return pd.DataFrame()

visitors_df = results_to_df(resp_unique_visitors)
visitors_df.head()

In [ ]:
if not visitors_df.empty:
    visitors_df["appId"] = visitors_df["appId"].astype(str)
    visitors_df["visitorId"] = visitors_df["visitorId"].astype(str)

    population_n = visitors_df["visitorId"].nunique()
    print("Population (unique active users):", population_n)
else:
    print("No rows returned")

In [ ]:
payload_daily_visitors = {
    "response": {"mimeType": "application/json"},
    "request": {
        "pipeline": [
            {
                "source": {
                    "timeSeries": {
                        "period": "dayRange",
                        "first": first_ms,
                        "count": -days,
                    },
                    "pageEvents": {"appId": TEST_APP_ID},
                }
            },
            {
                "group": {
                    "group": ["appId", "day", "visitorId"],
                    "fields": {
                        "eventCount": {"count": None},
                    },
                }
            },
            {"limit": 100000}
        ]
    },
}

resp_daily_visitors = endpoint.run(payload_daily_visitors)
daily_visitors_df = results_to_df(resp_daily_visitors)
daily_visitors_df.head()

In [ ]:
if not daily_visitors_df.empty:
    daily_visitors_df["visitorId"] = daily_visitors_df["visitorId"].astype(str)
    daily_visitors_df["day_dt"] = pd.to_datetime(daily_visitors_df["day"], unit="ms", utc=True)

    population_n = daily_visitors_df["visitorId"].nunique()
    print("Population (unique active users across window):", population_n)

    by_day = (
        daily_visitors_df.groupby("day_dt")["visitorId"]
        .nunique()
        .reset_index(name="daily_active_users")
        .sort_values("day_dt")
    )
    display(by_day)
else:
    print("No daily visitor rows returned")